In [1]:
# GPU 사용 가능 여부 확인
import sys
import torch
import random
import numpy as np
from ultralytics import RTDETR

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Python version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
PyTorch version: 2.7.1+cu118
CUDA version: 11.8
GPU available: True
GPU name: NVIDIA GeForce RTX 4090


In [2]:
# 모델 재현을 위한 랜덤시드 고정
def set_random_seed(seed_value=42):
    # Python의 기본 난수 시드 설정
    random.seed(seed_value)
    # NumPy 난수 시드 설정
    np.random.seed(seed_value)
    # PyTorch 난수 시드 설정 (CPU)
    torch.manual_seed(seed_value)
    # PyTorch 난수 시드 설정 (GPU)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    # CuDNN 설정
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed()

# matplotlib default 설정
import matplotlib.pyplot as plt
plt.style.use('default')

In [3]:
# 1. 모델 불러오기
model = RTDETR('rtdetr-l.pt')

model.info()

# train 파라미터 설정
train_params = {
    'data': "/home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/data.yaml",
    'epochs': 150,
    'imgsz': 640,
    'batch': 16,
    'project': './runs',
    'name': 'rtdert-l',
    'seed': 42,
    'optimizer': 'AdamW',           # 동일하게 AdamW 사용
    'weight_decay': 1e-4,
    'cos_lr': True,
    'warmup_epochs': int(0.05*150), # hf 설정과 유사하게 전체 epoch의 5%를 warmup에 사용
    'lr0': 0.0001,                   # 공식 권장값이 1e-3
    'lrf': 0.01,
    'workers': 12,                   # default: 8
    'patience': 40,

    # [데이터 증강 (HF와 공정한 비교를 위해 통제)]
    'hsv_h': 0.015,  # 참외 노란색 보존
    'hsv_s': 0.3,
    'hsv_v': 0.3,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.9,    # HF의 crop/affine 줌인/아웃과 유사한 강도
    'fliplr': 0.5,
    'mosaic': 0.0,   # 공정성을 위해 YOLO 필살기 끄기
    'mixup': 0.0,
    'erasing': 0.2
}

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs


In [4]:
# 모델 학습
results = model.train(**train_params)

Ultralytics 8.4.21 🚀 Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24091MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=rtdert-l, nbs=64, nms=False, opset=None, opti

/home/leedh/bin/miniconda3/envs/paper_exp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4080.7±2204.7 MB/s, size: 6956.6 KB)
val: Scanning /home/leedh/바탕화면/hf-object-detection-trainer/data/99_exp_total_dataset/labels/val.cache... 154 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 154/154 7.7Mit/s 0.0s
optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0001), 226 bias(decay=0.0)
Plotting labels to /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l/labels.jpg... 
Image sizes 640 train, 640 val
Using 12 dataloader workers
Logging results to /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l
Starting training for 150 epochs...

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      1/150        12G      2.959          5      1.141         74        640: 0% ──────────── 0/45  2.5s

/home/leedh/bin/miniconda3/envs/paper_exp/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      12.4G      1.765       1.38     0.7079         67        640: 100% ━━━━━━━━━━━━ 45/45 2.0it/s 22.4s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 9.2it/s 0.5s0.1s
                   all        154        755   0.000108    0.00662    5.5e-05   1.56e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150      12.4G      1.931     0.2499     0.4327         60        640: 0% ──────────── 0/45  0.2s

...

    118/150      12.4G     0.2949     0.3356    0.04258         82        640: 100% ━━━━━━━━━━━━ 45/45 4.4it/s 10.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 9.4it/s 0.5s0.1s
                   all        154        755      0.934      0.942      0.967      0.739

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    119/150      12.4G     0.2616     0.3417    0.06195         78        640: 0% ──────────── 0/45  0.2s

/home/leedh/bin/miniconda3/envs/paper_exp/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    119/150      12.4G     0.2709     0.3262    0.04005         56        640: 100% ━━━━━━━━━━━━ 45/45 4.4it/s 10.1s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 9.5it/s 0.5s0.1s
                   all        154        755      0.934      0.939      0.967      0.747

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    120/150      12.5G     0.3236     0.4024    0.02911         72        640: 0% ──────────── 0/45  0.2s

/home/leedh/bin/miniconda3/envs/paper_exp/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    120/150      12.5G     0.2848     0.3399     0.0452         48        640: 100% ━━━━━━━━━━━━ 45/45 4.4it/s 10.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 9.4it/s 0.5s0.1s
                   all        154        755      0.938      0.935       0.97      0.756
EarlyStopping: Training stopped early as no improvement observed in last 40 epochs. Best results observed at epoch 80, best model saved as best.pt.
To update EarlyStopping(patience=40) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

120 epochs completed in 0.390 hours.
Optimizer stripped from /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l/weights/last.pt, 66.3MB
Optimizer stripped from /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l/weights/best.pt, 66.3MB

Validating /home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l/weights/best

In [ ]:
# best 모델 불러오기
model = RTDETR('/home/leedh/바탕화면/hf-object-detection-trainer/exp/runs/detect/runs/rtdert-l/weights/best.pt')

model.info()

# validation 파라미터 설정
val_params = {
    'save_json': True
}

metrics = model.val()